# Getting the climate and soil data

Now that we have initial parameters for our species, we need to define the landscape in which the calibration will take place.

To adapt to the Python functions I've made for these Jupyter Notebooks (see [here](./functionsForCalibration.py)), simulations will take place on a single cell. 

What we need is :

- The climate file (containing all of the climate variables need by PnET)
- The ecoregion file and PnET ecoregion parameters (to decide which ones will be used during the calibration)
- Initial communities files (if needed)

## The climate file

See [Gustafson and Miranda (2023)](./ReferencesAndData/Documentation/Gustafson2024PnETUserGuide.pdf) for detailed information about what the climate file used in PnET simulations must contain.

[Gustafson and Miranda (2023)](./ReferencesAndData/Documentation/Gustafson2024PnETUserGuide.pdf) recommends using a constant climate for calibrating (p. 69). Gustafson also recommends using an "ideal" climate at some points, but deciding what is an ideal climate for a given species is a difficult proposition. As we are going to compare the growth curves generated by PnET Succession with the growth curves estimated from data of the National Forest Inventory (NFI) of Canada (see [](./5.Other_important_data_before_starting.ipynb)), the idea will be rather to get long-term monthly averages (as recommanded by [Gustafson and Miranda (2023)](./ReferencesAndData/Documentation/Gustafson2024PnETUserGuide.pdf) p. 73) in the study landscape.

PnET Succession requires 5 climate variables to function :

- Maximum Monthly temperature (°C)
- Minimum Monthly temperature (°C)
- Photosynthetically Active Radiation (umol/m2/s)
- Sum of precipitations during the month (mm)
- Mean monthly atmospheric CO2 concentration (ppm)

### Extent of the climate data

Our study landscape for this calibration (see [0.2.Goals_and_assumptions.ipynb](./0.2.Goals_and_assumptions.ipynb)) is located in Quebec, in the Mauricie region.

We will thus get climate data for this area. Since this is a large area, it will certainly cover several cells from the climate datasets we are going to use (see below). In addition, this area landscape two zones with a relatively different climate : warmer and more mixed forests in the south, and colder and more boreal forests in the north of the area.

In order to simplify things for this calibration, the climate stream we will use for our simulation will be the one of the climate cell closest to the center (centroid) of the study landscape. Therefore, the climate stream that we will generate here will not be averaged accoss space. But we will average it accross time (see below) in order to create "mild" and "ideal" conditions without extreme events, by averaging every day of the year accross a the historical period for which we have the data (i.e. every january 1st data for a given variable will the same, and will be the average of all january 1st in the original).

### Climate data source for 🌡️ temperature, 🌧️ precipitation and ☀ solar radiation

I did a lot of searching for a climate dataset that would be easy to access to make historical climate files for this calibration, but which would also be useful for "real" simulations once the calibration was done.

At first, I looked into the [ESGF Metagrid](https://esgf.github.io/software.html), which is (to my understanding) an enormous decentralized catalog of climate data (data generated by Global Climate Models by different research time throughout the world). There is an enormous trove of data here, and it can be accessed quite easily using the [intake-esgf](https://github.com/esgf2-us/intake-esgf) Python package. However, one needs to search in this catalog to find datasets with historical data, the right variables, the right time step, etc. It was quite hard, and not always very stable; this is because the datasets are hosted on "nodes" run by several universities throughout the world, and these would sometime shut down or be unavailable for a time.

Then, a dataset was very recently published (at the time of writing this), in December 2025, by Ouranos - a Quebec-based research consortium on climate which regularly produces climate dataset for Canada or for all of North america. This dataset, which had a higher resolution that most others, seemed perfect for our use.

Here, I describe the dataset from Ouranos; [see here for the article describing the dataset](https://doi.org/10.1038/s41597-025-06289-7).

#### What variables are available in the dataset ?

All variables we need for PnET-Succession are available in the dataset (see [this pdf](https://zenodo.org/records/17465777/files/variables_CRCM5_CMIP6.pdf?download=1) from the [Zenodo webpage of the dataset](https://zenodo.org/records/17465777) for the variable list) :

- tasmax => Daily or monthly
- tasmin => Daily or monthly
- rsds (downwelling shortwave radiation) => 1 hour or daily or monthly
- pr (precipitations, liquid or solid !) => 1 hour or daily daily or monthly

**The dataset also seem to contain all of the variables needed for BFOLDS succession** (wind speed and direction, relative humidity, etc.).

#### What is the resolution of the dataset, and its scenarios ?

Resolution : 0.11 degrees, about 12 km resolution, timestep of 5 minutes inside the model.

Simulations are produced at 0.11° horizontal grid spacing over North America from 1950 to 2100.

Up to four different emissions scenarios are used for future climate simulations (2015–2100).

Thefore, historical data is for 1950-2015.

#### What are the access points for the dataset ?

Two links on [their Zenodo page](https://zenodo.org/records/17465777) :

- Annual files : https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/birdhouse/disk2/ouranos/CORDEX/catalog.html

- Aggregated datasets : https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/datasets/simulations/RCM-CMIP6/catalog.html



#### What are the models selected for this ensemble, and why were they selected ?

Explanation of the datasets according to the paper (https://www.nature.com/articles/s41597-025-06289-7) :

> Simulations driven by GCMs from the Coupled Model Intercomparison Project 6 (CMIP6) cover years 1950–2014 for the historical period, while future climate simulations span 2015–2100. Future simulations follow several shared socioeconomic pathway (SSP) climate scenarios. SSP1-2.6 and SSP3-7.0 are required by CORDEX. Simulations following SSP2-4.5 are also run given interest in this scenario for climate adaptation. One simulation following the high-emissions SSP5-8.5 scenario has also been carried out.

> All simulations seem to have been done with only one climate model, called CRCM5. CRCM5 is not a Global Climate Model (GCM); it is a Regional Climate Model (RCM). RCMs allow for a higher resolution and to account for regional specificities; but they require climate conditions on the bounds of the area that is simulated. These climate conditions are thus taken from GCM - which simulate the entire earth. So, CRCM5 only simulates Canada, but needs data from GCMs to generate results. As such, we say that the RCM is "driven by" a given GCM.

> Although selecting only four models out of 56 may seem like a very small sample, most are not suitable for RCM application due to technical or compatibility constraints. Moreover, many of these models are interdependent (see16), which further limits the effective number of independent simulations to be selected. To elaborate further, models are selected based on several criteria, including data availability, model performance, climate sensitivity and compatibility with the CRCM5. The most important criterion is the availability of the data needed to drive the simulations. We require driving data for the historical simulation, which is an entry deck for CMIP6, and future simulations from the ScenarioMIP project. Data for SSP1-2.6, SSP2-4.5, and SSP3-7.0 must be available. As CRCM5 can handle only Gregorian or fixed 365-day calendars, models with a fixed 360-day calendar are excluded. 

They also justify that these 4 models tend to have different climate sensibility (for a given augmentation of atmospheric CO2 in a simulation, the models tend to generate more or less increase in temperature) : 

> We also attempted to span a wide range of equilibrium climate sensitivity (ECS) values, which represent the amount of long-term warming resulting from a doubling of atmospheric CO2 concentrations. The ECS values for our selected driving GCMs include 5.64 K for CanESM5, 4.8K for CNRM-ESM2-1, 3.0K for MPI-ESM-1-2-LR and 2.54 K for NorESM2-MM.

Other criteria :

> Additionally, more subjective selection crtieria include storage space required for the driving data (related to model resolution), collaborations with model developers (CanESM5), and use of the model by other regional modelling communities such as EURO-CORDEX (CNRM-ESM2-1). The possibility of using several members favored the selection of MPI-ESM-1-2-LR.

Performance of the models :

> Performance of the driving GCMs globally and over North America was also considered by comparing key driving variables to ERA5, including temperature, specific humidity, mean and variance of geopotential, sea surface temperature, and sea ice extent (e.g. Model Selection Dashboard. However, this has been used only for the selection of last GCM (NorESM2-MM).

Biases : 

The paper identifies several biaises, especially in greenland, southern Canada and Alberta/manitoba. Also precipitation biases with a east-west gradient (too much precipitation in the east, too much dryness in the west). These biases are calculated by comparing ERA5 versus CRCM5 drive by ERA5. Cold biases are also found by comparing ERA5 to CRCM5 driven by the 4 other models than ERA5. Biases in summer seem weaker. Strong precipitations biases in the oceanic zones ? Might be related to cyclone devellopment. But these are not so much present in Canada per se, more California and mexico.

CRCM5 driven by CanESM5 predicts the strongest warming, with CRCM5 driven NorESM2-MM predicting the weakest warming. 

> Although the high ECS of CanESM5 has been deemed unlikely (see 24), it remains useful as part of a narrative approach to examine potential outcomes in very warm scenarios for North America.

In the case of the 4 models given here, CNRM-ESM2-1 is the closest to the average equilibrium climate sensibility (ECS) of the 4 models (average is around 3.99K, MPI-ESM-1-2-LR has a value of 4.8K). MPI-ESM-1-2-LR is also a good choice. 

#### Which driving GCM to choose here ?

To simplify things during the calibration, we could choose only one stream of data from one of the driving GCMs. Criterias possible : the one with the most average equilibrium climate sensibility (ECS); the one that has the least bias / performs best on historical or verification data.

The main author/contact for the article, Dominique Paquin, recommanded that we use MPI as a driver model, as it seems to be the one that produces the most reliable data according to her (although she precised that using several models is always best; which is what we will do in the simulations after calibrating PnET-Succession.

:::{tip} Why we are using a single climate model (instead of several together) here
Climate scientists often recommand to use the results from several climate models (GCMs) at once so as to explore the influence of the variability in results that exists between models, and the incertainty that results from it.

Here, I choose to use the results from a single climate model for several reasons :

- The goal of this calibration is not to explore climate change predictions, but simply to calibrate the growth and dynamic of our tree species. For that, we will use historical data simulated by the model we chose (TaiESM1).
- This calibration is already quite complex, and I'd like to keep it as simple as possible.
- Using several models at once requires one of two things : either using a single climate file whose values are averaged along the values simulated by the different GCMs, which results in very, very "mild" climate conditions (since all extreme variations are not synchronized between models); or to use several climate files and create several PnET-Succession growth curves at each point (which makes things quite complicated).

However, once you have calibrated PnET Succession and will use it in your LANDIS-II simulations, it might be good to generate different climate files based on different climate models in order to create "climate replicates" for your LANDIS-II simulation that will allow you to generate an envellope of variability/uncertainty on your LANDIS-II results. To that end, you can use the scripts of this page to generate several climate files for different climate models available in ESGF (be warned that not all models have all of the variables you will need for all of the climate scenario you will want to explore). You can also use the functions from the Xclim python package that will allow you to select a smaller number of models that will represent your climate data properly (see [here](https://xclim.readthedocs.io/en/stable/apidoc/xclim.ensembles.html#xclim.ensembles._reduce.kmeans_reduce_ensemble); you will need to create a large ensemble of climate data to use these functions).
:::

## Generating temperature, precipitations, and radiation data from Ouranos CRCM5-CMIP6

In [5]:
# The code in this cell loads the functions needed to extract the climate data from the PAVIS database from Ouranos,
# using the CanDSC-M6 dataset
# We will use it to extract minimum and maximum monthly temp + total monthly precipitation for climate scenario SSP126
# The code in this cell will be re-used on other pages to generate the values for other climate scenarios
# Right now, it doesn't really matter since the first calibration step (monoculture in best conditions) will use
# historical data

# Importing the packages needed
# Siphon is used to connect to the data,
# xarray is used to manipulate the data objects
# The rest is for progress bars, utilities
# Clisops is used to subset the data based on a polygon
from functionsForCalibration import *
from siphon.catalog import TDSCatalog
import xarray as xr
from dask.diagnostics import ProgressBar
from IPython.display import display
from clisops.core import subset
import pandas as pd
import numpy as np
from tqdm import tqdm

# This function will help us define the URL to get the right dataset,
# corresponding to the right variable; the right SSP scenario; and the right model.
# This URL was made by browsing the THREDD server catalog of datasets, as was recommanded to me by email by the Canadian Centre for Climate Services (ECCC
# It points to the location of monthly maximum temperature data for CanDCS-M6, and climate scenario SSP126. The models are from CIMP6 from the IPCC.
def getURLDatasetHistoricalCRCM5CMIP6(GCM, replicate):
    url = ("https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/dodsC/datasets/simulations/RCM-CMIP6/CORDEX/NAM-12/mon/NAM-12_" +
           str(GCM) +
           "_historical_" + 
           str(replicate) + 
           "_OURANOS_CRCM5_v1-r1_mon_195001-201412.ncml")
    return url

# We define the time period of the data we want and prepare the dataframe
# that we will export for PnET - that will ultimatily contain a monthly measure
# of Tmin, Tmax, Precpitations, CO2 and PAR for each months of each year
# We will go up to 2015, the historical data limit in CRCM5-CMIP6
print("Initializing data ranges and objects...")
startYear = 1950
endYear = 2014
years = np.repeat(np.arange(startYear, endYear+1), 12)
# Create months 1-12 for each year
months = np.tile(np.arange(1, 13), (endYear+1)-startYear)
# Create DataFrame
df = pd.DataFrame({
    'Year': years,
    'Month': months
})
numberOfMonths = len(df["Month"])

print("Opening the dataset...")
# We load the data for the driving GCM decided in the previous cells, with
# its first replicate name (taken from the THREDDS server of ouranos)
ds = xr.open_dataset(getURLDatasetHistoricalCRCM5CMIP6("MPI-ESM1-2-LR",
                                                      "r1i1p1f1"))

# This prints information about the dataset
# display(ds)

print("Subsetting to the study landscape...")
# Subset the dataset by polygon
# What we get in ds1 is an array of grid cells with a third dimension corresponding to time
ds1 = subset.subset_shape(
    ds, shape="./ReferencesAndData/SpatialBoundaries/ManawanTerritoryExtent.shp"
) 

print("Dimensions of the dataset : " + str(ds1.dims))

#################################################
# MAXIMUM MONTHLY TEMPERATURE
#################################################
# We select the variable data and months we need - all of them
# 0 is january of 1950. numberOfMonths is the number of months from 1950, defined
# at the beginning of the cell.
print("Converting Maximum monthly temperature...")
ds_months = ds1["tasmax"].isel(time=slice(0, numberOfMonths))
# With this command, we can get the monthly averages across all cells in one very quick step
listOfTmaxMonthly = kelvin_to_celsius(ds_months.mean(dim=['rlat', 'rlon']).values.tolist())
df["Tmax"] = listOfTmaxMonthly

#################################################
# MINIMUM MONTHLY TEMPERATURE
#################################################
print("Converting Minimum monthly temperature...")
# We select the variable data and months we need - all of them
ds_months = ds1["tasmin"].isel(time=slice(0, numberOfMonths))
# With this command, we can get the monthly averages across all cells in one very quick step
listOfTminMonthly = kelvin_to_celsius(ds_months.mean(dim=['rlat', 'rlon']).values.tolist())
# print(ds1)
df["Tmin"] = listOfTminMonthly

#################################################
# PRECIPITATIONS
#################################################
print("Converting monthly precipitations...")
# We select the variable data and months we need - all of them
ds_months = ds1["pr"].isel(time=slice(0, numberOfMonths))
# With this command, we can get the monthly averages across all cells in one very quick step
# Precipitations are relative to a surface, so we can average them without issues.
listOfPrecTotMonthly = ds_months.mean(dim=['rlat', 'rlon']).values.tolist()
# Conversion of preciptations
# In the CRCM5-CMIP6 dataset, the units of precipitations are kg m-2 s-1 (for a second, but averaged other a whole month).
# See the metadata of ds_month for a validation of this.
# In PnET-Succession, the unit that we need depends on how we feed the climate stream to PnET-Succession :
# - If we give the climate stream directly to PnET-Succssion with its internal library,
#   the units is sum of precipitations for the month in mm (see PnET-Succession user guide)
# - If we give the climate stream through the LANDIS-II climate library, we can give it as
#   daily or monthly sum, but it is in cm (see the user guide of the climate library, table 1.
# Here, we prepare a file for the climate library of LANDIS-II; as such, we go from instannaeous fluxes in kg m-2 s-1
# to a monthly sum in cm.
# As such, to convert, we need to adapt to the changing amounts of seconds in a month (depends on the number of days in the month)
# We prepare a list of days in the month. This is a bit complex, so we use a custom function to do the trick.
# See docstring of convert_monthly_precip_to_cm() for more.
listOfPrecTotMonthly = convert_monthly_precip_to_cm(listOfPrecTotMonthly, (1950, 1), (2014, 12))
df["SumOfPrecipitations"] = listOfPrecTotMonthly

#################################################
# PHOTOSYNTHETICALLY ACTIVATE RADIATION (PAR)
#################################################
print("Converting rsds daily to daytime...")
# First, we have to convert rsds (shortwave downwelling solar radiation)
# from daily averaged values from daytime averaged values (before we transform it into PAR)
# Even if we're looking at rsds at a monthly timestep (monthly averaged), this average is done
# for the 24 hours of each day of the month.
# However, PnET-Succession deals with the daytime hours for the solar radiation (precisely, it needs 
# "Mean monthly value of Photosynthetically Active Radiation during daylight hours"), meaning that we
# have to account for the daytime (sunrise to sunset) of each day of the month, then get the monthly
# daytime, and then make the conversion of solar radiation from daily to daytime.
# This function takes care of it, and creates a new climate variable in our dataset for that.
ds1 = convert_to_daylight_average(ds1, "rsds")
# We take the shortwave downwelling solar radiation, named rsds
ds_months = ds1["rsds_MonthlyDaytime"].isel(time=slice(0, numberOfMonths))
# With this command, we can get the monthly averages across all cells in one very quick step
listOfRadiationMonthly = ds_months.mean(dim=['rlat', 'rlon']).values.tolist()

print("Converting rsds_daytime to PAR...")
# Transforming downwelling shortwave radiation (rsds) in W/m2 to photosynthetically activate radiation PAR in umol.m2/s-1
# Based on the PnET User Guide's instructions
# Downwelling shortwave radiation (rsds which we have here) is often refered as global solar radiation.
# See https://library.wmo.int/viewer/68695/?offset=3#page=298&viewer=picture&o=search&n=0&q=shortwave . But other references exist.
# So, Downwelling shortave radiation is for wavelengths of 0.2–4.0 μm; but PAR is for 0.4–0.7 μm (lesser range).
# As such, to convert our Downwelling Shortwave Radiation in W/m2 to PAR in umol.m2/s-1, 
# we must multiply it by 2.02 as indicated in the user guide of PnET Succession. See the guide for more info.
listOfRadiationMonthly = [x * 2.02 for x in listOfRadiationMonthly]
df["PAR"] = listOfRadiationMonthly

Initializing data ranges and objects...
Opening the dataset...
Subsetting to the study landscape...
Dimensions of the dataset : FrozenMappingWarningOnValuesAccess({'rlat': 20, 'rlon': 22, 'bounds': 4, 'time': 780, 'bnds': 2})
Converting Maximum monthly temperature...
Converting Minimum monthly temperature...
Converting monthly precipitations...
Converting rsds daily to daytime...
Processing 780 time steps, 20 latitudes, 22 longitudes


Processing months: 100%|█████████████████████████████████████████████████████████████| 780/780 [03:50<00:00,  3.38it/s]


Calculating daylight averages...
Done!
Converting rsds_daytime to PAR...


In [4]:
#################################################
# EXPORT TO CSV
#################################################
# We output the dataframe to a .csv
# df.to_csv("./ReferencesAndData/dataFrameClimate_Manawan_Ouranos.csv", sep=',', index=False, encoding='utf-8')

# print(df)


# We rename the columns to match the names that the climate library of LANDIS-II expect
# PAR already has the right name
df = df.rename(columns={'tasmin': 'Tmin', 'tasmax': 'Tmax', 'SumOfPrecipitations':'precip'})

df["Ecoregion"] = "eco1"

# We print the result
print(df)

##################################
# Giving the dataframe the right columns
# for the LANDIS-II v8 climate library
##################################

# Create a unique identifier for each lat/lon pair
# mergedDataframe['lat_lon'] = mergedDataframe['lat'].astype(str) + '_' + mergedDataframe['lon'].astype(str)

# Melt the dataframe to convert variables into rows
melted_df = pd.melt(
    df, 
    id_vars=['Year', 'Month', "Ecoregion"],
    value_vars=['Tmin', 'Tmax', "PAR", "precip"],
    var_name='Variable',
    value_name='value'
)

# Pivot the table to get lat_lon as columns (which can then be used as ecoregions)
result_df = melted_df.pivot_table(
    index=['Year', 'Month', 'Variable'],
    columns='Ecoregion',
    values='value'
).reset_index()

# Convert the column index from MultiIndex to regular Index
result_df.columns.name = None
result_df = result_df.sort_values(by=['Variable', 'Year', 'Month'])

# We replace the name of the column that has the lat_lon pairing into an 
# ecoregion name (the one used in our PnET one cell scenario, eco1)
# result_df = result_df.rename(columns={result_df.columns[-1]: 'eco1'})
# print(result_df)

# We print as .csv
result_df.to_csv("./ReferencesAndData/Climate Data/dataFrameClimate_historicalMonthly_Ouranos.csv", sep=',', index=False, encoding='utf-8')
print(result_df)

     Year  Month       Tmax       Tmin     precip          PAR Ecoregion
0    1950      1  -5.373328 -15.161230   8.186317   220.704819      eco1
1    1950      2  -8.705023 -18.739066   8.076399   395.376224      eco1
2    1950      3  -2.022192 -14.746924   7.016513   633.388979      eco1
3    1950      4   3.820520  -7.170599   7.462637   912.260132      eco1
4    1950      5  13.236017   2.782831  10.993223  1025.980425      eco1
..    ...    ...        ...        ...        ...          ...       ...
775  2014      8  21.738824  11.503076  12.180150   920.335307      eco1
776  2014      9  16.575647   7.186151  16.403019   645.717363      eco1
777  2014     10   8.705103   2.218927  19.146976   339.002393      eco1
778  2014     11   0.962579  -5.533423   9.024735   236.495038      eco1
779  2014     12  -8.425970 -16.830725   4.783478   210.506408      eco1

[780 rows x 7 columns]
      Year  Month Variable         eco1
0     1950      1      PAR   220.704819
4     1950      2   

Now, we have to add the CO2. We're going to reuse old code I've made for a previous climate data source (ESGF); the CO2 data comes from https://zenodo.org/records/5021361 .

:::{hint} Why are we adding CO2 data from a different dataset or model than the one used for the climate variables ?
From my small understanding and knowledge of Climate models (including Global Climate Models or GCMs, or Regional Climate Models like CRCM5 that we're using here), the CO2 atmospheric concentration is most often treated in these models as a prescribed variable rather than an output variables. This means that the evolution of the values of the CO2 atmospheric concentration in these models is actually determined before the simulation, and is not influenced by the model during the simulation. The evolution is actually prescribed by the emission scenario chosen : the SSPs or RCPs scenarios. 

Some rare climate models include feedbacks between human emissions of CO2 and atmospheric CO2 concentration, for example with vegetation feedbacks (see [HadGEM2-ES](https://gmd.copernicus.org/articles/4/543/2011/gmd-4-543-2011.pdf#:~:text=HadGEM2-ES%20also%20represents,with%20atmospheric%20aerosols.). In these models, the CO2 atmospheric concentration is therefore another output of the model, and is influenced by the model and not completly fixed before the simulation (only human CO2 emissions are fixed by the emission scenario).

But since CO2 can still change seasonaly and locally (because of the effects of forests, etc.), and because CO2 is one of the variables driving photosynthesis in PnET-Succession, I thought it would be best to have a CO2 dataset that changes with time and with space, even if it's from another model. Here, that's what we're using to add CO2 data to our climate file. It's not super important for the calibration of PnET-Succession, and we could have just used yearly averages of CO2 atmospheric concentrations derived from historical data. But I was looking forward to future simulations that would use future data, and thought it best to do things right. Therefore, **you can re-use this code to generate future CO2 concentration data for your study landscape after you're done with this calibration.**
:::

In [1]:
# First, we download the CO2 data from https://zenodo.org/records/5021361 

from functionsForCalibration import *
import pandas as pd
import urllib.request
import subprocess
import os

# We use the zenodo_get library to download files from Zenodo very fast; other methods tend to be slow.
# See https://github.com/dvolgyes/zenodo_get . It is installed in the Docker image.
# WARNING : will take around 1.5GB of space for the data from 1950 to 2150 for a full climate scenario.

# We download the file with historical data
os.system("cd ReferencesAndData && zenodo_get 5021361 -g CO2_1deg_month_1850-2013.nc")

# We can also download data for future conditions, like this :
# os.system("cd ReferencesAndData && zenodo_get 5021361 -g CO2_SSP126_2015_2150.nc")

Title: Global monthly distributions of atmospheric CO2 concentrations under the historical and future scenarios
Keywords: Atmospheric chemistry, Atmospheric dynamics, Climate and Earth system modelling
Publication date: 2021-06-23
DOI: 10.5281/zenodo.5021361
Total size: 1.0 GB

Link: https://zenodo.org/records/5021361/files/CO2_1deg_month_1850-2013.nc   size: 1.0 GB


15

In [11]:
from functionsForCalibration import *

historical_ds = xr.open_dataset("./ReferencesAndData/CO2_1deg_month_1850-2013.nc")
# future_ds = xr.open_dataset("./ReferencesAndData/CO2_SSP126_2015_2150.nc")

# Shapefile to select where we take our values; if several climate cells are in the shapefile,
# then the CO2 values in all cells will be averaged (see function process_co2_data)
shapefile = "./ReferencesAndData/SpatialBoundaries/ManawanTerritoryExtent.shp"

# We load the dataframe
# We prepare the dataframe in a format that matches the older function I made
# When I was using monthly data and LANDIS-II v7.
# We load the file created previously
df_climate = pd.read_csv("./ReferencesAndData/Climate Data/dataFrameClimate_historicalMonthly_Ouranos.csv")
# Convert to year month day to string
df_climate['date_string'] = (df_climate['Year'].astype(str) + '-' + 
                     df_climate['Month'].astype(str))
# Get unique dates
uniqueDates = df_climate['date_string'].unique()
# Create the new dataframe
dfToFill = pd.DataFrame(columns=["Year", "Month", "Variable"])
# Put the dates in
dfToFill["date_string"] = uniqueDates
# extract the dates correctly
dfToFill['Year'] = dfToFill["date_string"].str.split('-').str[0].astype(int)
dfToFill['Month'] = dfToFill["date_string"].str.split('-').str[1].astype(int)
# Add variable name
dfToFill["Variable"] = ["CO2"] * len(dfToFill)
# Remove the date_string column
dfToFill.drop('date_string', axis=1, inplace=True)
df_climate.drop('date_string', axis=1, inplace=True)
# Now, the dataframe is ready to be filled with CO2 values with the old function !
print(dfToFill)

dfToFill = process_co2_data_monthly(historical_ds, shapefile, dfToFill)

# Display the updated dataframe
print(dfToFill)
print(dfToFill.head())
print(dfToFill.tail())

     Year  Month Variable
0    1950      1      CO2
1    1950      2      CO2
2    1950      3      CO2
3    1950      4      CO2
4    1950      5      CO2
..    ...    ...      ...
775  2014      8      CO2
776  2014      9      CO2
777  2014     10      CO2
778  2014     11      CO2
779  2014     12      CO2

[780 rows x 3 columns]
You are outside of historical data! Use process_co2_data_withFutureInterpolation to deal with future data from https://zenodo.org/records/5021361
For now, will use values from the latest year to deal with this row
You are outside of historical data! Use process_co2_data_withFutureInterpolation to deal with future data from https://zenodo.org/records/5021361
For now, will use values from the latest year to deal with this row
You are outside of historical data! Use process_co2_data_withFutureInterpolation to deal with future data from https://zenodo.org/records/5021361
For now, will use values from the latest year to deal with this row
You are outside of his

In [12]:
# Finally, we add this CO2 data to the previous data, and re-average it if needed

# First, we have to rename the CO2 column in the dataframe we just filled so that it corresponds to an ecoregion now
# (this is due to the difference in the climate library formats with LANDIS-II v8 and the old functions I was using before switching)
# The column name CO2_Concentration is created by the function process_co2_data
dfToFill = dfToFill.rename(columns={'CO2_Concentration': 'eco1'})

# We concatenate the dataframes
df_climate = pd.concat([df_climate, dfToFill], ignore_index=True)

# We check if needed
print(df_climate)

# We save everything
df_climate.to_csv("./ReferencesAndData/Climate Data/dataFrameClimate_historicalMonthly_Ouranos.csv", sep=',', index=False, encoding='utf-8')

# Now that everything is done, we can remove the CO2 files downloaded
os.remove("./ReferencesAndData/CO2_1deg_month_1850-2013.nc")
# os.remove("./ReferencesAndData/CO2_SSP126_2015_2150.nc")

      Year  Month Variable         eco1
0     1950      1      PAR   220.704819
1     1950      2      PAR   395.376224
2     1950      3      PAR   633.388979
3     1950      4      PAR   912.260132
4     1950      5      PAR  1025.980425
...    ...    ...      ...          ...
3895  2014      8      CO2   395.650866
3896  2014      9      CO2   400.600555
3897  2014     10      CO2   404.031146
3898  2014     11      CO2   406.003131
3899  2014     12      CO2   406.856389

[3900 rows x 4 columns]


We will end up by making a "Spinup" climate file. This file will simply contain the averages of the measures between 1950 and 1980, and will then distribute these averages from the year 1700 to 1951. These will serve for spinup procedures to initialize the biomass of the cohorts in our simulations.

In [1]:
import pandas as pd

# Read the climate data
df = pd.read_csv("./ReferencesAndData/Climate Data/dataFrameClimate_historicalMonthly_Ouranos.csv")

# Filter data for years 1950-1980
df_filtered = df[(df['Year'] >= 1950) & (df['Year'] <= 1980)]

# Calculate monthly averages for each variable
monthly_averages = df_filtered.groupby(['Month', 'Variable'])['eco1'].mean().reset_index()

# Create date range from 1700-01 to 1951-12
years = range(1800, 1952)
months = range(1, 13)
date_combinations = [(year, month) for year in years for month in months]

# Build new dataframe
new_data = []
for year, month in date_combinations:
    for _, row in monthly_averages[monthly_averages['Month'] == month].iterrows():
        new_data.append({
            'Year': year,
            'Month': month,
            'Variable': row['Variable'],
            'eco1': row['eco1']
        })

# Create and save new dataframe
df_new = pd.DataFrame(new_data)
df_new = df_new.sort_values(by=['Variable', 'Year', 'Month'], ascending=[True, True, True])
print(df_new)
df_new.to_csv("./ReferencesAndData/Climate Data/dataFrameClimate_SpinupMonthly_Ouranos.csv", index=False)

print(f"Created extended dataset with {len(df_new)} rows")
print(f"Date range: {df_new['Year'].min()}-{df_new['Year'].max()}")


      Year  Month Variable        eco1
0     1800      1      CO2  327.163384
5     1800      2      CO2  327.771870
10    1800      3      CO2  327.148786
15    1800      4      CO2  324.769317
20    1800      5      CO2  320.943121
...    ...    ...      ...         ...
9099  1951      8   precip   14.856778
9104  1951      9   precip   13.913735
9109  1951     10   precip   13.369664
9114  1951     11   precip   10.279178
9119  1951     12   precip    8.139437

[9120 rows x 4 columns]
Created extended dataset with 9120 rows
Date range: 1800-1951


We finish by creating some averaged monthly inputs from our monthly measurements. We do this because [Eric Gustafson recommands in his calibration tips](./ReferencesAndData/Documentation/Gustafson2024PnETUserGuide.pdf) to use averaged climate historical inputs for the calibration simulations. This is because averaged climate inputs tend to remove the extreme events from the climate data (e.g. very hot summers, very cold winters, droughts, etc.) and makes the climate "mild", effectively removing the effect of extreme or even variable climate events on our calibration simulations. This is quite important.

The function `average_climate_data_by_month()` used here will create `.csv` files with a suffix `_MonthlyAveraged` that we can then use.

In [1]:
from functionsForCalibration import *

average_climate_data_by_month("./ReferencesAndData/Climate Data/dataFrameClimate_historicalMonthly_Ouranos.csv")
average_climate_data_by_month("./ReferencesAndData/Climate Data/dataFrameClimate_SpinupMonthly_Ouranos.csv")

'./ReferencesAndData/Climate Data/dataFrameClimate_SpinupMonthly_Ouranos_MonthlyAveraged.csv'

## Soils

Gustafon recommands SILO (Silty Loam/Silt Loam) soil from the default PnET Succession SaxtonAndRawls file (`C:\Program Files\LANDIS-II-v7\extensions\Defaults\SaxtonAndRawlsParameters.txt`) as a soil that retains water well. This is because the first calibration step should use an "ideal soil" where water is well-retained so that lack of water (or watterlogging) does not influence growth at this calibration step.

We can use the SILO soil type easily by modifying the SoilType of `eco1` in the PnET One Cell Scenario that we will use (see `SimulationFiles/PnETGitHub_OneCellSim_v8/`). So, nothing more to do here.

## CLUES FOR FUTURE DATA

- Alex Chubaty might have scripts to generate initial data in BC and Alberta, and even other provinces ?
- Caren Dymonc for initial data in BC ?
- Yan Boulanger : BioSIM climate data for small scale/downscaled climate data for PnET + BFOLDS ? Needed so that we can have PnET Climate input that change according to slope and topography in particular. Will need to create many smaller ecoregions that are combo of soils + climate "zones" in the area + classes of slopes/aspect. BioSIM might help in generating those things.